# Cleaning Messy HR Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

In [2]:
df = pd.read_csv("messy_HR_data.csv")

df.head(2)

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
0,grace,25,50000,Male,HR,Manager,"April 5, 2018",D,email@example.com,NaN
1,david,NaN,65000,Female,Finance,Director,2020/02/20,F,user@domain.com,123-456-7890


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               1000 non-null   object
 1   Age                841 non-null    object
 2   Salary             1000 non-null   object
 3   Gender             1000 non-null   object
 4   Department         1000 non-null   object
 5   Position           1000 non-null   object
 6   Joining Date       1000 non-null   object
 7   Performance Score  1000 non-null   object
 8   Email              610 non-null    object
 9   Phone Number       815 non-null    object
dtypes: object(10)
memory usage: 78.3+ KB


In [4]:
#Replace Nan Values as np.nan
df.replace(['nan', 'NAN', '', ' '], np.nan, inplace=True)

df.head(3)

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
0,grace,25,50000,Male,HR,Manager,"April 5, 2018",D,email@example.com,NaN
1,david,NaN,65000,Female,Finance,Director,2020/02/20,F,user@domain.com,123-456-7890
2,hannah,35,SIXTY THOUSAND,Female,Sales,Director,01/15/2020,C,email@example.com,098-765-4321


In [5]:
# trim all string
for col in ['Name', 'Gender', 'Department', 'Position', 'Email', 'Phone Number']:
    df[col] = df[col].str.strip()


### Fix Age

In [6]:
df.Age.unique()

array(['25', nan, '35', '40', 'thirty', '50'], dtype=object)

In [7]:
df['Age'] = df['Age'].replace({'thirty': 30})
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age'].unique()

array([25., nan, 35., 40., 30., 50.])

### Fix Salary

In [8]:
df['Salary'].unique()

array(['50000', '65000', 'SIXTY THOUSAND', ' NAN ', '70000', '55000'],
      dtype=object)

In [9]:
import re

def salary_to_numeric(s):
    if pd.isna(s):
        return np.nan
    s = str(s).upper().replace(',', '').strip()
    if s == "SIXTY THOUSAND":
        return 60000
    # Remove non-digit characters
    s = re.sub(r'[^0-9]', '', s)
    return int(s) if s else np.nan

df['Salary'] = df['Salary'].apply(salary_to_numeric)

df['Salary'].unique()

array([50000., 65000., 60000.,    nan, 70000., 55000.])

### Fix Joining Date

In [10]:
df['Joining Date'].unique()

array(['April 5, 2018', '2020/02/20', '01/15/2020', '03-25-2019',
       '2019.12.01'], dtype=object)

In [11]:
from datetime import datetime
import pandas as pd
import numpy as np

def parse_date(date_str):
    if pd.isna(date_str):
        return np.nan
    date_str = str(date_str).strip()
    formats = ["%B %d, %Y", "%Y/%m/%d", "%m/%d/%Y", "%m-%d-%Y", "%Y.%m.%d"]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    # if none match
    return np.nan


### Fix Joining Date
df['Joining Date'] = df['Joining Date'].apply(parse_date)
df['Joining Date'].unique()

<DatetimeArray>
['2018-04-05 00:00:00', '2020-02-20 00:00:00', '2020-01-15 00:00:00',
 '2019-03-25 00:00:00', '2019-12-01 00:00:00']
Length: 5, dtype: datetime64[ns]

### Enocode Performance Score

In [12]:
df['Performance Score'].unique()

array(['D', 'F', 'C', 'A', 'B'], dtype=object)

In [13]:
performance_encoder = LabelEncoder()

df['Performance Score'] = performance_encoder.fit_transform(df['Performance Score'])

df['Performance Score'].unique()

array([3, 4, 2, 0, 1])

In [14]:
df.head(3)

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
0,grace,25.0,50000.0,Male,HR,Manager,2018-04-05,3,email@example.com,NaN
1,david,NaN,65000.0,Female,Finance,Director,2020-02-20,4,user@domain.com,123-456-7890
2,hannah,35.0,60000.0,Female,Sales,Director,2020-01-15,2,email@example.com,098-765-4321


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Name               1000 non-null   object        
 1   Age                841 non-null    float64       
 2   Salary             833 non-null    float64       
 3   Gender             1000 non-null   object        
 4   Department         1000 non-null   object        
 5   Position           1000 non-null   object        
 6   Joining Date       1000 non-null   datetime64[ns]
 7   Performance Score  1000 non-null   int64         
 8   Email              610 non-null    object        
 9   Phone Number       624 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 78.3+ KB


In [16]:
# Save dataset
df.to_csv("Dirty HR Dataset.csv")

## Imputation of Salary Column using Median
Replacing null values with median of Department Salary

In [17]:
df['Salary'] = df.groupby('Department')['Salary'].transform(lambda x: x.fillna(x.median()))

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Name               1000 non-null   object        
 1   Age                841 non-null    float64       
 2   Salary             1000 non-null   float64       
 3   Gender             1000 non-null   object        
 4   Department         1000 non-null   object        
 5   Position           1000 non-null   object        
 6   Joining Date       1000 non-null   datetime64[ns]
 7   Performance Score  1000 non-null   int64         
 8   Email              610 non-null    object        
 9   Phone Number       624 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 78.3+ KB


## Imputation of Age Column using KNN Imputation

In [19]:
df.Age.unique()

array([25., nan, 35., 40., 30., 50.])

In [20]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

df_encoded = df.copy()
df_encoded['Gender'] = df_encoded['Gender'].astype('category').cat.codes
df_encoded['Department'] = df_encoded['Department'].astype('category').cat.codes
df_encoded['Position'] = df_encoded['Position'].astype('category').cat.codes

knn_imputer = KNNImputer(n_neighbors=5)

df_encoded['Age'] = knn_imputer.fit_transform(df_encoded[['Age', 'Salary', 'Gender', 'Department', 'Position']])[:, 0]

df['Age'] = df_encoded['Age']


In [21]:
df.head(3)

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
0,grace,25.0,50000.0,Male,HR,Manager,2018-04-05,3,email@example.com,NaN
1,david,31.0,65000.0,Female,Finance,Director,2020-02-20,4,user@domain.com,123-456-7890
2,hannah,35.0,60000.0,Female,Sales,Director,2020-01-15,2,email@example.com,098-765-4321


In [22]:
df.drop_duplicates()

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
0,grace,25.0,50000.0,Male,HR,Manager,2018-04-05,3,email@example.com,NaN
1,david,31.0,65000.0,Female,Finance,Director,2020-02-20,4,user@domain.com,123-456-7890
2,hannah,35.0,60000.0,Female,Sales,Director,2020-01-15,2,email@example.com,098-765-4321
3,eve,32.0,50000.0,Female,IT,Manager,2018-04-05,0,name@company.org,NaN
4,grace,40.0,60000.0,Female,Finance,Manager,2020-01-15,4,name@company.org,098-765-4321
...,...,...,...,...,...,...,...,...,...,...
995,jack,50.0,65000.0,Female,HR,Manager,2020-02-20,4,NaN,098-765-4321
996,jack,30.0,50000.0,Male,Finance,Analyst,2018-04-05,2,NaN,555-555-5555
997,hannah,30.0,70000.0,Male,IT,Assistant,2020-01-15,3,user@domain.com,NaN
998,bob,25.0,65000.0,Other,Marketing,Manager,2018-04-05,3,email@example.com,NaN


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Name               1000 non-null   object        
 1   Age                1000 non-null   float64       
 2   Salary             1000 non-null   float64       
 3   Gender             1000 non-null   object        
 4   Department         1000 non-null   object        
 5   Position           1000 non-null   object        
 6   Joining Date       1000 non-null   datetime64[ns]
 7   Performance Score  1000 non-null   int64         
 8   Email              610 non-null    object        
 9   Phone Number       624 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 78.3+ KB


## Impute Age and Salary using EM Algorithm

In [24]:
df = pd.read_csv("Dirty HR Dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1000 non-null   int64  
 1   Name               1000 non-null   object 
 2   Age                841 non-null    float64
 3   Salary             833 non-null    float64
 4   Gender             1000 non-null   object 
 5   Department         1000 non-null   object 
 6   Position           1000 non-null   object 
 7   Joining Date       1000 non-null   object 
 8   Performance Score  1000 non-null   int64  
 9   Email              610 non-null    object 
 10  Phone Number       624 non-null    object 
dtypes: float64(2), int64(2), object(7)
memory usage: 86.1+ KB


In [25]:
def em_impute_age_salary(df, max_iter=100, tol=1e-4): 
    data = df[['Age', 'Salary']].copy()
    
    # Initial imputation (mean imputation)
    mu = data.mean()
    data_filled = data.fillna(mu)

    for iteration in range(max_iter):
        mu_old = mu.copy()

        # M-step: estimate mean and covariance
        mu = data_filled.mean().values
        sigma = np.cov(data_filled.T)

        # E-step: update missing values conditionally
        for i in range(len(data)):
            row = data.iloc[i]
            missing = row.isna().values

            if missing.any() and not missing.all():
                observed = ~missing

                sigma_oo = sigma[np.ix_(observed, observed)]
                sigma_mo = sigma[np.ix_(missing, observed)]

                mu_o = mu[observed]
                mu_m = mu[missing]

                x_o = data_filled.iloc[i, observed].values

                # Conditional expectation
                x_m = mu_m + sigma_mo @ np.linalg.inv(sigma_oo) @ (x_o - mu_o)

                data_filled.iloc[i, missing] = x_m

        # Convergence check
        if np.linalg.norm(mu - mu_old) < tol:
            break

    # Replace original columns
    df[['Age', 'Salary']] = data_filled
    return df


# Usage
df = em_impute_age_salary(df)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1000 non-null   int64  
 1   Name               1000 non-null   object 
 2   Age                1000 non-null   float64
 3   Salary             1000 non-null   float64
 4   Gender             1000 non-null   object 
 5   Department         1000 non-null   object 
 6   Position           1000 non-null   object 
 7   Joining Date       1000 non-null   object 
 8   Performance Score  1000 non-null   int64  
 9   Email              610 non-null    object 
 10  Phone Number       624 non-null    object 
dtypes: float64(2), int64(2), object(7)
memory usage: 86.1+ KB
